# Resize Dataset Images to 1024 × 1024

This notebook completes the reconstructed dataset after running
`00_Convert_Original_Annotations.ipynb`.

The annotation converter writes per-image JSON files whose coordinates are
already expressed on a **1024 × 1024** canvas. This notebook finds the
corresponding original images, resizes them physically to the same dimensions,
and saves them beside the converted JSON annotations.

It does **not** alter the converted JSON files.

Expected input:

```text
<original_dataset_root>/
├── fa/
│   ├── train/
│   ├── val/          # when available
│   └── test/
├── fca/
└── fcb/

<converted_dataset_root>/
├── fa/
│   ├── train/json/
│   ├── val/json/
│   └── test/json/
├── fca/
└── fcb/
```

Expected output:

```text
<converted_dataset_root>/
├── fa/
│   ├── train/
│   │   ├── images/   # resized 1024 × 1024 images
│   │   └── json/     # existing converted annotations
│   ├── val/
│   └── test/
├── fca/
└── fcb/
```


In [ ]:
# ============================================================
# USER SETTINGS — FILL THE PATHS BEFORE RUNNING
# ============================================================

# Root folder of the original downloaded handwritten-diagram datasets
ORIGINAL_DATASET_ROOT = ""

# Root folder created by 00_Convert_Original_Annotations.ipynb
# The resized images will be written into this same folder beside the JSON files.
CONVERTED_DATASET_ROOT = ""

# Dataset families to process
FAMILIES_TO_PROCESS = ["fa", "fca", "fcb"]

# Use None to process every split containing a json/ folder.
# Example for a restricted run: ["train", "val", "test"]
SPLITS_TO_PROCESS = None

TARGET_WIDTH = 1024
TARGET_HEIGHT = 1024

# Existing correctly sized images are skipped when False.
OVERWRITE_EXISTING = False

# Number of parallel image workers. Use 1 for sequential execution.
WORKERS = 4


## 1. Imports

The notebook uses Pillow for image loading and high-quality resizing. Images are
resized directly to 1024 × 1024 because the converted annotation coordinates
use the same fixed canvas.


In [ ]:
from __future__ import annotations

import json
import os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

from PIL import Image, ImageOps

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

print(f"Pillow version: {Image.__version__}")


## 2. Path and image-resolution helpers

The normal source location is:

```text
<original root>/<family>/<split>/<image filename>
```

A recursive fallback is included so that minor differences in the downloaded
repository layout do not require code changes. Ambiguous duplicate filenames
are reported instead of guessed.


In [ ]:
SUPPORTED_IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"
}


def require_nonempty_path(value: str, variable_name: str) -> Path:
    value = str(value).strip()
    if not value:
        raise ValueError(
            f"{variable_name} is blank. Fill it in the USER SETTINGS cell before running."
        )
    path = Path(value).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"{variable_name} does not exist: {path}")
    return path


def discover_converted_splits(
    converted_root: Path,
    families: Sequence[str],
    requested_splits: Optional[Sequence[str]] = None,
) -> List[Tuple[str, str, Path]]:
    """Return (family, split, json_dir) entries that actually exist."""
    discovered: List[Tuple[str, str, Path]] = []

    for family in families:
        family_dir = converted_root / family
        if not family_dir.exists():
            print(f"Skipping family '{family}': not found at {family_dir}")
            continue

        if requested_splits is None:
            candidate_split_dirs = sorted(
                path for path in family_dir.iterdir() if path.is_dir()
            )
        else:
            candidate_split_dirs = [family_dir / split for split in requested_splits]

        for split_dir in candidate_split_dirs:
            json_dir = split_dir / "json"
            if json_dir.exists() and any(json_dir.glob("*.json")):
                discovered.append((family, split_dir.name, json_dir))
            elif requested_splits is not None:
                print(
                    f"Skipping {family}/{split_dir.name}: no converted JSON files at {json_dir}"
                )

    return discovered


def build_source_image_index(search_root: Path) -> Dict[str, List[Path]]:
    """
    Index source images by both exact filename and lowercase stem.

    Lists are retained so duplicate names can be detected safely.
    """
    index: Dict[str, List[Path]] = defaultdict(list)

    for path in search_root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_IMAGE_EXTENSIONS:
            continue

        index[path.name.lower()].append(path)
        index[path.stem.lower()].append(path)

    return dict(index)


def resolve_source_image(
    original_root: Path,
    family: str,
    split: str,
    image_name: str,
    split_index: Dict[str, List[Path]],
    family_index: Dict[str, List[Path]],
) -> Path:
    """Resolve one source image without silently choosing an ambiguous duplicate."""
    expected = original_root / family / split / image_name
    if expected.exists():
        return expected

    filename_key = Path(image_name).name.lower()
    stem_key = Path(image_name).stem.lower()

    for index, scope_name in (
        (split_index, f"{family}/{split}"),
        (family_index, family),
    ):
        candidates = index.get(filename_key, [])
        if not candidates:
            candidates = index.get(stem_key, [])

        unique_candidates = sorted(set(candidates))
        if len(unique_candidates) == 1:
            return unique_candidates[0]
        if len(unique_candidates) > 1:
            formatted = "\n".join(f"  - {path}" for path in unique_candidates)
            raise RuntimeError(
                f"Ambiguous source image '{image_name}' inside {scope_name}:\n{formatted}"
            )

    raise FileNotFoundError(
        f"Could not find source image '{image_name}' for {family}/{split}."
    )


## 3. Image resizing

The aspect ratio is intentionally **not preserved**. Every image is mapped
directly to 1024 × 1024 because the converted node boxes, endpoints, and arrow
paths were scaled independently along the horizontal and vertical axes to this
same canvas.

The original filename and extension are preserved so that the JSON `image`
field continues to refer to the correct file.


In [ ]:
def choose_resampling(width: int, height: int, target_width: int, target_height: int):
    """Use Lanczos when reducing an image and bicubic interpolation otherwise."""
    if width > target_width or height > target_height:
        return Image.Resampling.LANCZOS
    return Image.Resampling.BICUBIC


def save_resized_image(
    source_path: Path,
    destination_path: Path,
    target_width: int,
    target_height: int,
    overwrite: bool,
) -> Dict[str, object]:
    """
    Resize and save one image.

    Returns a serializable status dictionary for the final report.
    """
    if destination_path.exists() and not overwrite:
        try:
            with Image.open(destination_path) as existing:
                if existing.size == (target_width, target_height):
                    return {
                        "status": "skipped_existing",
                        "source": str(source_path),
                        "destination": str(destination_path),
                        "original_size": list(existing.size),
                        "output_size": list(existing.size),
                    }
        except Exception:
            # A corrupt or unreadable destination should be regenerated.
            pass

    destination_path.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(source_path) as image:
        image = ImageOps.exif_transpose(image)
        original_size = image.size

        resample = choose_resampling(
            width=original_size[0],
            height=original_size[1],
            target_width=target_width,
            target_height=target_height,
        )
        resized = image.resize((target_width, target_height), resample=resample)

        suffix = destination_path.suffix.lower()
        save_kwargs: Dict[str, object] = {}

        # JPEG does not support alpha channels or palette transparency.
        if suffix in {".jpg", ".jpeg"} and resized.mode not in {"L", "RGB"}:
            resized = resized.convert("RGB")
            save_kwargs = {"quality": 95, "subsampling": 0}

        # Preserve binary/grayscale/RGB modes for the diagram images whenever possible.
        resized.save(destination_path, **save_kwargs)

    return {
        "status": "resized",
        "source": str(source_path),
        "destination": str(destination_path),
        "original_size": list(original_size),
        "output_size": [target_width, target_height],
    }


def read_annotation_image_name(json_path: Path) -> str:
    with json_path.open("r", encoding="utf-8") as handle:
        record = json.load(handle)

    image_name = str(record.get("image", "")).strip()
    if not image_name:
        raise ValueError(f"Missing or empty 'image' field in {json_path}")

    declared_size = record.get("size", {})
    declared_width = int(declared_size.get("w", 0))
    declared_height = int(declared_size.get("h", 0))

    if (declared_width, declared_height) != (TARGET_WIDTH, TARGET_HEIGHT):
        raise ValueError(
            f"Annotation canvas mismatch in {json_path}: "
            f"{declared_width} × {declared_height}, expected "
            f"{TARGET_WIDTH} × {TARGET_HEIGHT}."
        )

    return image_name


## 4. Split-level processing

Only JSON files produced by the annotation-conversion notebook are treated as
the authoritative sample list. This prevents unrelated source images from being
copied into the reconstructed dataset.


In [ ]:
def process_converted_split(
    original_root: Path,
    converted_root: Path,
    family: str,
    split: str,
    json_dir: Path,
    target_width: int,
    target_height: int,
    overwrite: bool,
    workers: int,
) -> Dict[str, object]:
    json_files = sorted(json_dir.glob("*.json"))
    output_image_dir = converted_root / family / split / "images"
    output_image_dir.mkdir(parents=True, exist_ok=True)

    normal_split_root = original_root / family / split
    split_search_root = (
        normal_split_root if normal_split_root.exists() else original_root / family
    )
    family_search_root = original_root / family

    if not family_search_root.exists():
        raise FileNotFoundError(
            f"Original dataset family was not found: {family_search_root}"
        )

    split_index = build_source_image_index(split_search_root)
    if split_search_root == family_search_root:
        family_index = split_index
    else:
        family_index = build_source_image_index(family_search_root)

    tasks: List[Tuple[Path, Path, Path]] = []
    preparation_errors: List[Dict[str, str]] = []

    for json_path in json_files:
        try:
            image_name = read_annotation_image_name(json_path)
            source_path = resolve_source_image(
                original_root=original_root,
                family=family,
                split=split,
                image_name=image_name,
                split_index=split_index,
                family_index=family_index,
            )
            destination_path = output_image_dir / image_name
            tasks.append((json_path, source_path, destination_path))
        except Exception as exc:
            preparation_errors.append(
                {
                    "json": str(json_path),
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

    results: List[Dict[str, object]] = []
    execution_errors: List[Dict[str, str]] = []

    def run_one(task: Tuple[Path, Path, Path]) -> Dict[str, object]:
        json_path, source_path, destination_path = task
        result = save_resized_image(
            source_path=source_path,
            destination_path=destination_path,
            target_width=target_width,
            target_height=target_height,
            overwrite=overwrite,
        )
        result["json"] = str(json_path)
        return result

    if workers <= 1:
        iterator = tasks
        if tqdm is not None:
            iterator = tqdm(tasks, desc=f"{family}/{split}", unit="image")

        for task in iterator:
            try:
                results.append(run_one(task))
            except Exception as exc:
                execution_errors.append(
                    {
                        "json": str(task[0]),
                        "source": str(task[1]),
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )
    else:
        with ThreadPoolExecutor(max_workers=workers) as executor:
            future_to_task = {
                executor.submit(run_one, task): task for task in tasks
            }
            completed = as_completed(future_to_task)
            if tqdm is not None:
                completed = tqdm(
                    completed,
                    total=len(future_to_task),
                    desc=f"{family}/{split}",
                    unit="image",
                )

            for future in completed:
                task = future_to_task[future]
                try:
                    results.append(future.result())
                except Exception as exc:
                    execution_errors.append(
                        {
                            "json": str(task[0]),
                            "source": str(task[1]),
                            "error": f"{type(exc).__name__}: {exc}",
                        }
                    )

    status_counts: Dict[str, int] = defaultdict(int)
    for result in results:
        status_counts[str(result["status"])] += 1

    summary = {
        "family": family,
        "split": split,
        "json_files": len(json_files),
        "prepared_tasks": len(tasks),
        "status_counts": dict(status_counts),
        "preparation_errors": preparation_errors,
        "execution_errors": execution_errors,
        "output_image_dir": str(output_image_dir),
    }

    with (converted_root / family / split / "image_resize_summary.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(summary, handle, indent=2)

    return summary


## 5. Final verification

The verification step checks that every converted annotation has one readable
image of exactly 1024 × 1024 and that no unexpected image files were introduced
into each output split.


In [ ]:
def verify_reconstructed_dataset(
    converted_root: Path,
    discovered_splits: Sequence[Tuple[str, str, Path]],
    target_width: int,
    target_height: int,
) -> Dict[str, object]:
    split_reports: List[Dict[str, object]] = []
    total_json = 0
    total_images = 0
    missing_images: List[str] = []
    invalid_dimensions: List[Dict[str, object]] = []
    unreadable_images: List[Dict[str, str]] = []
    unexpected_images: List[str] = []

    for family, split, json_dir in discovered_splits:
        image_dir = converted_root / family / split / "images"
        json_files = sorted(json_dir.glob("*.json"))
        expected_names: set[str] = set()

        for json_path in json_files:
            try:
                expected_names.add(read_annotation_image_name(json_path))
            except Exception as exc:
                unreadable_images.append(
                    {
                        "path": str(json_path),
                        "error": f"Annotation error: {type(exc).__name__}: {exc}",
                    }
                )

        existing_image_paths = sorted(
            path
            for path in image_dir.glob("*")
            if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
        )
        existing_names = {path.name for path in existing_image_paths}

        for expected_name in sorted(expected_names):
            image_path = image_dir / expected_name
            if not image_path.exists():
                missing_images.append(str(image_path))
                continue

            try:
                with Image.open(image_path) as image:
                    if image.size != (target_width, target_height):
                        invalid_dimensions.append(
                            {
                                "path": str(image_path),
                                "actual": list(image.size),
                                "expected": [target_width, target_height],
                            }
                        )
            except Exception as exc:
                unreadable_images.append(
                    {
                        "path": str(image_path),
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )

        for extra_name in sorted(existing_names - expected_names):
            unexpected_images.append(str(image_dir / extra_name))

        split_reports.append(
            {
                "family": family,
                "split": split,
                "annotations": len(json_files),
                "expected_images": len(expected_names),
                "existing_images": len(existing_image_paths),
            }
        )
        total_json += len(json_files)
        total_images += len(existing_image_paths)

    return {
        "target_size": [target_width, target_height],
        "total_annotations": total_json,
        "total_output_images": total_images,
        "missing_images": missing_images,
        "invalid_dimensions": invalid_dimensions,
        "unreadable_files": unreadable_images,
        "unexpected_images": unexpected_images,
        "splits": split_reports,
        "is_complete": not (
            missing_images
            or invalid_dimensions
            or unreadable_images
            or unexpected_images
        ),
    }


## 6. Run image reconstruction

Fill the two paths at the top of the notebook, then run this cell. The final
report is saved as `image_resize_validation_report.json` in the converted
dataset root.


In [ ]:
# ============================================================
# RUN THE IMAGE RESIZING
# ============================================================

original_root = require_nonempty_path(
    ORIGINAL_DATASET_ROOT, "ORIGINAL_DATASET_ROOT"
)
converted_root = require_nonempty_path(
    CONVERTED_DATASET_ROOT, "CONVERTED_DATASET_ROOT"
)

if original_root == converted_root:
    raise ValueError(
        "ORIGINAL_DATASET_ROOT and CONVERTED_DATASET_ROOT must be different."
    )

if TARGET_WIDTH <= 0 or TARGET_HEIGHT <= 0:
    raise ValueError("TARGET_WIDTH and TARGET_HEIGHT must be positive integers.")

workers = max(1, int(WORKERS))

discovered_splits = discover_converted_splits(
    converted_root=converted_root,
    families=FAMILIES_TO_PROCESS,
    requested_splits=SPLITS_TO_PROCESS,
)

if not discovered_splits:
    raise RuntimeError(
        "No converted split containing JSON annotations was found. "
        "Run 00_Convert_Original_Annotations.ipynb first."
    )

print("Discovered converted splits:")
for family, split, json_dir in discovered_splits:
    print(f"  - {family}/{split}: {len(list(json_dir.glob('*.json')))} annotations")

all_summaries: List[Dict[str, object]] = []

for family, split, json_dir in discovered_splits:
    summary = process_converted_split(
        original_root=original_root,
        converted_root=converted_root,
        family=family,
        split=split,
        json_dir=json_dir,
        target_width=TARGET_WIDTH,
        target_height=TARGET_HEIGHT,
        overwrite=OVERWRITE_EXISTING,
        workers=workers,
    )
    all_summaries.append(summary)

validation_report = verify_reconstructed_dataset(
    converted_root=converted_root,
    discovered_splits=discovered_splits,
    target_width=TARGET_WIDTH,
    target_height=TARGET_HEIGHT,
)

final_report = {
    "settings": {
        "target_width": TARGET_WIDTH,
        "target_height": TARGET_HEIGHT,
        "overwrite_existing": OVERWRITE_EXISTING,
        "workers": workers,
    },
    "resize_summaries": all_summaries,
    "validation": validation_report,
}

report_path = converted_root / "image_resize_validation_report.json"
with report_path.open("w", encoding="utf-8") as handle:
    json.dump(final_report, handle, indent=2)

print("\n" + "=" * 72)
print("IMAGE RESIZING COMPLETE")
print("=" * 72)
print(f"Annotations checked : {validation_report['total_annotations']}")
print(f"Output images       : {validation_report['total_output_images']}")
print(f"Missing images      : {len(validation_report['missing_images'])}")
print(f"Invalid dimensions  : {len(validation_report['invalid_dimensions'])}")
print(f"Unreadable files    : {len(validation_report['unreadable_files'])}")
print(f"Unexpected images   : {len(validation_report['unexpected_images'])}")
print(f"Dataset complete    : {validation_report['is_complete']}")
print(f"Validation report   : {report_path}")

if not validation_report["is_complete"]:
    print(
        "\nThe reconstructed dataset is not yet complete. "
        "Inspect the validation report before training."
    )
else:
    print(
        "\nThe images and converted annotations now share the same "
        "1024 × 1024 coordinate system."
    )
